# GraphQL sobre una base de grafos (Neo4j) — Linaje de modelos

Tutorial de la **Sesión 2** de MLOps II. Combina lo visto en la clase: exponemos por **GraphQL** (Strawberry + FastAPI) el **linaje** de un modelo, y el resolver lo obtiene con **Cypher** desde **Neo4j**.

**GraphQL** es la puerta flexible (el cliente pide la forma exacta); **Neo4j** es el motor de relaciones (recorre el grafo de linaje).

### Aprenderás a
* Levantar Neo4j con Docker y sembrar un grafo de linaje.
* Escribir un servidor GraphQL cuyo resolver corre Cypher.
* Consultar el linaje de un modelo con una sola query GraphQL.

## 1. Requisitos

* **[uv](https://docs.astral.sh/uv/)** y **Docker Desktop**. Python ≥ 3.11 (lo exige Strawberry).
* Dependencias (desde `clase2/Practica/`):

```bash
uv pip install strawberry-graphql fastapi "uvicorn[standard]" requests neo4j
```

Si corres este notebook, usa el **kernel del entorno de uv** (ver *Puesta en marcha* del README raíz).

In [ ]:
# Si te falta alguna dependencia en el kernel actual, descomenta:
# !uv pip install strawberry-graphql fastapi "uvicorn[standard]" requests neo4j

## 2. Levantar Neo4j con Docker

Ejecuta esto en una **terminal aparte** (deja el contenedor corriendo):

```bash
docker run --name neo4j-tp -p 7474:7474 -p 7687:7687 \
    -e NEO4J_AUTH=neo4j/testpass neo4j:latest
```

* UI del navegador: <http://localhost:7474> (login `neo4j` / `testpass`)
* Driver bolt: puerto `7687`

Para frenar/borrar el contenedor: `docker stop neo4j-tp && docker rm neo4j-tp`.

## 3. El servidor GraphQL

La siguiente celda escribe el archivo `graphql_neo4j_lineage.py`. El esquema tiene un tipo `Model` con un campo `lineage`, cuyo **resolver ejecuta Cypher** contra Neo4j.

In [ ]:
%%writefile graphql_neo4j_lineage.py
"""GraphQL sobre Neo4j — linaje de modelos (Sesión 2, MLOps II)."""
from __future__ import annotations
import os, sys
from typing import List, Optional

import strawberry
from fastapi import FastAPI
from strawberry.fastapi import GraphQLRouter
from neo4j import GraphDatabase

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "testpass")

_driver = None
def get_driver():
    """Crea el driver de Neo4j una sola vez (no por request)."""
    global _driver
    if _driver is None:
        _driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    return _driver

# --- Esquema GraphQL ---
@strawberry.type
class Artifact:
    name: str
    kind: str  # Dataset | Feature | Experiment | Model | Deployment

@strawberry.type
class Model:
    name: str

    @strawberry.field
    def lineage(self) -> List[Artifact]:
        """Artefactos aguas arriba de este modelo (su linaje)."""
        cypher = """
        MATCH (a)-[:DERIVES*1..]->(m:Model {name: $name})
        RETURN DISTINCT a.name AS name, head(labels(a)) AS kind
        """
        with get_driver().session() as ses:
            rows = ses.run(cypher, name=self.name)
            return [Artifact(name=r["name"], kind=r["kind"]) for r in rows]

@strawberry.type
class Query:
    @strawberry.field
    def model(self, name: str) -> Optional[Model]:
        cypher = "MATCH (m:Model {name: $name}) RETURN m.name AS name"
        with get_driver().session() as ses:
            rec = ses.run(cypher, name=name).single()
            return Model(name=rec["name"]) if rec else None

schema = strawberry.Schema(query=Query)
app = FastAPI(title="Linaje por GraphQL sobre Neo4j")
app.include_router(GraphQLRouter(schema), prefix="/graphql")

# --- Sembrar el grafo de linaje de ejemplo ---
SEED = [
    "MATCH (n) DETACH DELETE n",
    "CREATE (d:Dataset {name:'transacciones_v3'})",
    "CREATE (f:Feature {name:'features_churn'})",
    "CREATE (e:Experiment {name:'exp_42'})",
    "CREATE (m:Model {name:'churn@3'})",
    "CREATE (dep:Deployment {name:'/v1/predict'})",
    "MATCH (d:Dataset),(f:Feature) CREATE (d)-[:DERIVES]->(f)",
    "MATCH (f:Feature),(e:Experiment) CREATE (f)-[:DERIVES]->(e)",
    "MATCH (e:Experiment),(m:Model) CREATE (e)-[:DERIVES]->(m)",
    "MATCH (m:Model),(dep:Deployment) CREATE (m)-[:DERIVES]->(dep)",
]
def seed() -> None:
    with get_driver().session() as ses:
        for stmt in SEED:
            ses.run(stmt)
    print("Grafo de linaje sembrado.")

if __name__ == "__main__":
    if "--seed" in sys.argv:
        seed()


## 4. Sembrar el grafo de linaje

Con Neo4j corriendo, creamos el grafo de ejemplo: `Dataset → Feature → Experiment → Model → Deployment`, unidos por relaciones `DERIVES`.

In [ ]:
from graphql_neo4j_lineage import seed
seed()   # requiere que el contenedor Neo4j esté corriendo

Puedes verificar el grafo en la UI de Neo4j (<http://localhost:7474>) con:

```cypher
MATCH (n)-[r]->(m) RETURN n, r, m
```

## 5. Levantar la API GraphQL

`uvicorn` bloquea el kernel, así que **levanta la API en una terminal aparte**:

```bash
uv run uvicorn graphql_neo4j_lineage:app --reload --port 8000
```

Queda GraphiQL en <http://localhost:8000/graphql>.

## 6. Consultar el linaje — cliente Python

Con la API corriendo, pedimos el linaje del modelo `churn@3` con **una sola query GraphQL**. El resolver traduce esa query a Cypher sobre Neo4j.

In [ ]:
import requests

URL = "http://localhost:8000/graphql"
query = """
query {
  model(name: "churn@3") {
    name
    lineage { name kind }
  }
}
"""
r = requests.post(URL, json={"query": query})
print(r.status_code)
print(r.json())

## 7. Explorar en GraphiQL

En <http://localhost:8000/graphql> pega la misma query y usa **Docs** para explorar el esquema. Prueba pedir solo `name` o solo `lineage { kind }`: la misma API responde exactamente lo que pidas.

### Cierre
* **GraphQL** expuso el linaje con un contrato tipado y flexible.
* **Neo4j** resolvió el recorrido de relaciones (`DERIVES*`) que en SQL serían muchos JOINs.
* Este es el patrón de **metadatos y linaje** de la plataforma del curso. En el **Mini-TP 2** lo aplicas a tu modelo.